# GCP Pose Estimation — Kaggle Training
GPU + Internet must be enabled. This is the safety gate: 3-epoch smoke first, eyeball predictions, then full run.

In [ ]:
# 1. Code + deps
!git clone <YOUR_REPO_URL> gcp
%cd gcp
!pip -q install -r requirements.txt

In [ ]:
# 2. Data (gdown the shared folder). Paths in configs/config.yaml expect:
#    data/gcp_marks.json, data/train_dataset/, data/test_dataset/
!gdown --folder 1RDNiAO1EynKrRDomcVNXQW1-ct5zzvE5 -O data/ --remaining-ok
!ls data

**Important:** after download, make sure `cfg.paths.label_file`, `train_dir`, `test_dir` in `configs/config.yaml` point at the actual downloaded subfolders. Adjust the paths if the folder names differ.

In [ ]:
# 3. SMOKE: 3 epochs, then inspect predictions schema before committing GPU time
import yaml
c = yaml.safe_load(open('configs/config.yaml')); c['training']['num_epochs'] = 3
yaml.safe_dump(c, open('configs/config.yaml', 'w'))
!python train.py --config configs/config.yaml
!python predict.py --config configs/config.yaml --checkpoint outputs/best.pt --output predictions.json
import json; p = json.load(open('predictions.json'))
k = next(iter(p)); print(k, '->', p[k]); print('total:', len(p))

In [ ]:
# 4. FULL run: restore 40 epochs
c = yaml.safe_load(open('configs/config.yaml')); c['training']['num_epochs'] = 40
yaml.safe_dump(c, open('configs/config.yaml', 'w'))
!python train.py --config configs/config.yaml

In [ ]:
# 5. Final predictions + download artifacts
!python predict.py --config configs/config.yaml --checkpoint outputs/best.pt --output predictions.json
import pandas as pd; pd.read_csv('outputs/training_log.csv').tail()

Download `outputs/best.pt`, `outputs/training_log.csv`, and `predictions.json`. Put `best.pt` on Drive and paste the link + the final PCK/F1 into `README.md`.